# Session 3 - Files, string processing and for loops
## Worksheet 3 - Stretch Goals

These are additional, more challenging exercises which you may like to attempt. They do not form part of the core module material so don't worry if you don't have time to complete them.

You will need to open this notebook from within the folder containing the files:
- dnaseq.txt
- dnaseq2.txt

## Stretch goals related to CRISPR
You can copy your code for the CRISPR design challenge in the earlier worksheet then adapt it to meet the additional requirements.

1. DNA is double stranded, so we also want to identify potential binding sites on the other strand. To do this you need to search for the reverse complement of NGG (CCN), then take the 20 nucleotides directly after that. Remember to reverse complement your sequence afterwards or the oligos won't work!

In [6]:
# Read DNA sequence file
file = open('dnaseq.txt', 'r')
dnaseq = file.read().strip()
file.close()

print(f"DNA Sequence: {dnaseq}")
print(f"Sequence Length: {len(dnaseq)} bases\n")

# Define complement base mapping
complement = {'a': 't', 't': 'a', 'c': 'g', 'g': 'c'}

# Function: Get reverse complement sequence
def reverse_complement(seq):
    # Reverse the sequence and get complement for each base
    return ''.join(complement[base] for base in seq[::-1])

# Find PAM sequences on the forward strand (NGG)
forward_grna_pam_pairs = []

# Loop through sequence looking for GG patterns (forward strand)
for i in range(len(dnaseq) - 2):
    if dnaseq[i+1] == 'g' and dnaseq[i+2] == 'g':
        pam_sequence = dnaseq[i] + dnaseq[i+1] + dnaseq[i+2]
        
        # Only keep PAMs with at least 20 upstream nucleotides
        if i >= 20:
            # Extract gRNA sequence (20 nucleotides upstream of PAM)
            grna_start = i - 20
            grna_sequence = dnaseq[grna_start:i]
            
            forward_grna_pam_pairs.append((grna_start, grna_sequence, pam_sequence))

# Find PAM sequences on the reverse strand (CCN)
reverse_grna_pam_pairs = []

# Loop through sequence looking for CC patterns (reverse strand PAM)
for i in range(len(dnaseq) - 2):
    if dnaseq[i] == 'c' and dnaseq[i+1] == 'c':
        # PAM on reverse strand is CCN (reverse complement of NGG)
        pam_sequence_reverse_strand = dnaseq[i] + dnaseq[i+1] + dnaseq[i+2]
        
        # Only keep PAMs with at least 20 downstream nucleotides
        if i + 23 <= len(dnaseq):
            # Extract sequence after PAM (20 nucleotides downstream)
            target_start = i + 3  # Start after the PAM
            target_end = target_start + 20
            target_sequence = dnaseq[target_start:target_end]
            
            # Get reverse complement to get the actual gRNA sequence
            grna_sequence = reverse_complement(target_sequence)
            pam_sequence = reverse_complement(pam_sequence_reverse_strand)
            
            reverse_grna_pam_pairs.append((i, grna_sequence, pam_sequence))

# Display forward strand results
if forward_grna_pam_pairs:
    print("Forward Strand gRNA-PAM Pairs:")
    print("PAM Position\tgRNA Sequence\t\t\tPAM Sequence\tFull Target")
    print("-" * 85)
    
    for grna_start, grna_seq, pam_seq in forward_grna_pam_pairs:
        full_target = grna_seq + " " + pam_seq
        pam_position = grna_start + 20  # PAM starts right after gRNA
        print(f"{pam_position}\t\t{grna_seq}\t{pam_seq}\t{full_target}")
else:
    print("No PAM sequences with ≥20 upstream nucleotides found on forward strand.")

# Display reverse strand results
if reverse_grna_pam_pairs:
    print("\nReverse Strand gRNA-PAM Pairs:")
    print("PAM Position\tgRNA Sequence\t\t\tPAM Sequence\tFull Target")
    print("-" * 85)
    
    for pam_position, grna_seq, pam_seq in reverse_grna_pam_pairs:
        full_target = grna_seq + " " + pam_seq
        print(f"{pam_position}\t\t{grna_seq}\t{pam_seq}\t{full_target}")
        
    print("\nDetailed Information for Reverse Strand gRNAs:")
    print("=" * 70)
    for i, (pam_position, grna_seq, pam_seq) in enumerate(reverse_grna_pam_pairs, 1):
        print(f"Reverse Strand gRNA {i}:")
        print(f"  - PAM Position on Forward Strand: {pam_position}")
        print(f"  - gRNA Sequence (20nt): {grna_seq}")
        print(f"  - PAM Sequence (3nt): {pam_seq}")
        print(f"  - Full Target Sequence: {grna_seq} {pam_seq}")
        print()
else:
    print("No PAM sequences with ≥20 downstream nucleotides found on reverse strand.")

# Summary of results
total_grnas = len(forward_grna_pam_pairs) + len(reverse_grna_pam_pairs)
print(f"\nTotal gRNAs Identified: {total_grnas}")
print(f"- Forward Strand: {len(forward_grna_pam_pairs)} gRNAs")
print(f"- Reverse Strand: {len(reverse_grna_pam_pairs)} gRNAs")


DNA Sequence: tagctggactgacctatatctggtcgttaggccgtttaaagcatcatggactacagact
Sequence Length: 59 bases

Forward Strand gRNA-PAM Pairs:
PAM Position	gRNA Sequence			PAM Sequence	Full Target
-------------------------------------------------------------------------------------
20		tagctggactgacctatatc	tgg	tagctggactgacctatatc tgg
28		ctgacctatatctggtcgtt	agg	ctgacctatatctggtcgtt agg
46		ttaggccgtttaaagcatca	tgg	ttaggccgtttaaagcatca tgg

Reverse Strand gRNA-PAM Pairs:
PAM Position	gRNA Sequence			PAM Sequence	Full Target
-------------------------------------------------------------------------------------
12		acggcctaacgaccagatat	agg	acggcctaacgaccagatat agg
31		gtagtccatgatgctttaaa	cgg	gtagtccatgatgctttaaa cgg

Detailed Information for Reverse Strand gRNAs:
Reverse Strand gRNA 1:
  - PAM Position on Forward Strand: 12
  - gRNA Sequence (20nt): acggcctaacgaccagatat
  - PAM Sequence (3nt): agg
  - Full Target Sequence: acggcctaacgaccagatat agg

Reverse Strand gRNA 2:
  - PAM Position on Forwar

2. If you are trying to disrupt a protein coding sequence by non-homologous end joining (basically error prone repair of a double straind break which usually inserts/deletes a few nucleotides resulting in a frame shift) then you need to search for sites within the open reading frame (ORF). The file dnaseq2.txt contains a sequence, the co-ordinates of the reading frame are indices 380-490. Search for any viable gRNA binding sites within the ORF, on both the forward and reverse strands

In [12]:
# Read DNA sequence file
file = open('dnaseq2.txt', 'r')
dnaseq = file.read().strip()
file.close()

print(f"DNA Sequence: {dnaseq}")
print(f"Sequence Length: {len(dnaseq)} bases")
print(f"ORF Coordinates: 380-490 (length: {490-380+1} bases)\n")

# Define complement base mapping
complement = {'a': 't', 't': 'a', 'c': 'g', 'g': 'c'}

# Function: Get reverse complement sequence
def reverse_complement(seq):
    return ''.join(complement[base] for base in seq[::-1])

# Define ORF region
orf_start = 380
orf_end = 490
orf_sequence = dnaseq[orf_start:orf_end+1]  # +1 to include the end position

print(f"ORF Sequence: {orf_sequence}\n")

# Find PAM sequences on the forward strand (NGG) within the ORF
forward_grna_pam_pairs = []

# Loop through ORF region looking for GG patterns (forward strand)
for i in range(orf_start, orf_end - 1):  # -1 to ensure we don't go out of range
    if i + 2 > orf_end:  # Skip if PAM would extend beyond ORF
        continue
        
    if dnaseq[i+1] == 'g' and dnaseq[i+2] == 'g':
        pam_sequence = dnaseq[i] + dnaseq[i+1] + dnaseq[i+2]
        
        # Check if there are at least 20 nucleotides upstream within the ORF
        if i - orf_start >= 20:
            # Extract gRNA sequence (20 nucleotides upstream of PAM)
            grna_start = i - 20
            grna_sequence = dnaseq[grna_start:i]
            
            # Verify the gRNA is entirely within the ORF
            if grna_start >= orf_start:
                forward_grna_pam_pairs.append((i, grna_sequence, pam_sequence))

# Find PAM sequences on the reverse strand (CCN) within the ORF
reverse_grna_pam_pairs = []

# Loop through ORF region looking for CC patterns (reverse strand PAM)
for i in range(orf_start, orf_end - 1):
    if i + 2 > orf_end:  # Skip if PAM would extend beyond ORF
        continue
        
    if dnaseq[i] == 'c' and dnaseq[i+1] == 'c':
        # PAM on reverse strand is CCN (reverse complement of NGG)
        pam_sequence_reverse_strand = dnaseq[i] + dnaseq[i+1] + dnaseq[i+2]
        
        # Check if there are at least 20 nucleotides downstream within the ORF
        if orf_end - (i + 2) >= 20:
            # Extract sequence after PAM (20 nucleotides downstream)
            target_start = i + 3  # Start after the PAM
            target_end = target_start + 20
            
            # Verify the target sequence is entirely within the ORF
            if target_end <= orf_end + 1:  # +1 because ORF end is inclusive
                target_sequence = dnaseq[target_start:target_end]
                
                # Get reverse complement to get the actual gRNA sequence
                grna_sequence = reverse_complement(target_sequence)
                pam_sequence = reverse_complement(pam_sequence_reverse_strand)
                
                reverse_grna_pam_pairs.append((i, grna_sequence, pam_sequence))

# Display forward strand results
if forward_grna_pam_pairs:
    print("Forward Strand gRNA-PAM Pairs Within ORF:")
    print("PAM Position\tgRNA Sequence\t\t\tPAM Sequence\tFull Target")
    print("-" * 85)
    
    for pam_position, grna_seq, pam_seq in forward_grna_pam_pairs:
        full_target = grna_seq + " " + pam_seq
        print(f"{pam_position}\t\t{grna_seq}\t{pam_seq}\t{full_target}")
else:
    print("No viable gRNA binding sites found on forward strand within ORF.")

# Display reverse strand results
if reverse_grna_pam_pairs:
    print("\nReverse Strand gRNA-PAM Pairs Within ORF:")
    print("PAM Position\tgRNA Sequence\t\t\tPAM Sequence\tFull Target")
    print("-" * 85)
    
    for pam_position, grna_seq, pam_seq in reverse_grna_pam_pairs:
        full_target = grna_seq + " " + pam_seq
        print(f"{pam_position}\t\t{grna_seq}\t{pam_seq}\t{full_target}")
else:
    print("No viable gRNA binding sites found on reverse strand within ORF.")

# Summary of results
total_grnas = len(forward_grna_pam_pairs) + len(reverse_grna_pam_pairs)
print(f"\nTotal gRNAs Identified Within ORF: {total_grnas}")
print(f"- Forward Strand: {len(forward_grna_pam_pairs)} gRNAs")
print(f"- Reverse Strand: {len(reverse_grna_pam_pairs)} gRNAs")

# Additional information about ORF disruption
if total_grnas > 0:
    print("\nORF Disruption Strategy:")
    print("These gRNAs can be used to introduce double-strand breaks within the coding sequence.")
    print("When repaired by error-prone non-homologous end joining (NHEJ), this typically")
    print("results in small insertions or deletions that cause frameshifts and disrupt the protein.")
    
    # Calculate approximate cutting positions within ORF
    print("\nApproximate cutting positions within ORF:")
    for pam_position, grna_seq, pam_seq in forward_grna_pam_pairs:
        cut_position = pam_position - 3  # Cas9 cuts ~3 bp upstream of PAM
        orf_cut_position = cut_position - orf_start
        print(f"Forward gRNA: Cut at position {cut_position} (ORF position {orf_cut_position})")
    
    for pam_position, grna_seq, pam_seq in reverse_grna_pam_pairs:
        # For reverse strand, the cut position is similar but on the opposite strand
        cut_position = pam_position + 3  # Approximate cut position
        orf_cut_position = cut_position - orf_start
        print(f"Reverse gRNA: Cut at position {cut_position} (ORF position {orf_cut_position})")


DNA Sequence: cgcgcgcccctccatggattaaccaaggggggttttgagtcaactcttcagacatcatttttcagtaccttttagcagatgcagacagtccccgcttcattcggggatccgcctctaactctcatgggtaatacttccaggctcttcctaaaaccacggcctccgccactttacacatactttaaacagcaagacctacgcatgaccctaacctcaacccataaggctgtttaatcatcgttctacaagggcgacttcatgcgttagctattgtgtcgtggctctcgtttcgcacgggcaaccggccagtgctgatagagtgctaagagcggctctcttactgacgacgttaagccgtccaaggcgcaactattagcctatacttaccccatctttcctcgggagggcccgagcgaggactcagcgtatcattagactcgttttgagattcgcatgctcgttagaaattatagtcacagttcataacggcgaaggagccgggtcgtcctcgcggtttgcgcttttgcctacacgtcgcaagacgatgcaattcggtgttctatggggtcgcgtgtttcgcgtgacggaagggtgaatatatagaggggactatgtacttcggcactctcgctccccgaacctcgatcgcgaagagacgccacgtagcgtgacgtcggcccttacccgagagtagacccatgccaactcctgccggacactaatgagccaagcggtcacgatcttgatcgtaggtatatagttgcgatacggagttgccaaatgccggcgttctagccatcacacgagttgtgcttcgttgacgcagaagtcgaatcttaagatcgcatttcagacgatccgttctcccaactacgtactgaacgttccttatcgggaaccaccgagcaggttagccggcttccatgatcccaaagctatcccgctcgcacgggagttaaaccgaatatccggcagatcaaaacact

3. The GC content should be in the range 40-60% for optimal performance. Check if this requirement is satisfied for the first of the gRNA sequences identified.

In [14]:
# Read DNA sequence file
file = open('dnaseq2.txt', 'r')
dnaseq = file.read().strip()
file.close()

# Define complement base mapping
complement = {'a': 't', 't': 'a', 'c': 'g', 'g': 'c'}

# Function: Get reverse complement sequence
def reverse_complement(seq):
    return ''.join(complement[base] for base in seq[::-1])

# Function: Calculate GC content percentage
def calculate_gc_content(sequence):
    g_count = sequence.count('G')
    c_count = sequence.count('C')
    total_bases = len(sequence)
    gc_content = (g_count + c_count) / total_bases * 100
    return gc_content

# Define ORF region
orf_start = 380
orf_end = 490

# Find PAM sequences on the forward strand (NGG) within the ORF
forward_grna_pam_pairs = []

# Loop through ORF region looking for GG patterns (forward strand)
for i in range(orf_start, orf_end - 1):
    if i + 2 > orf_end:
        continue
        
    if dnaseq[i+1] == 'g' and dnaseq[i+2] == 'g':
        pam_sequence = dnaseq[i] + dnaseq[i+1] + dnaseq[i+2]
        
        # Check if there are at least 20 nucleotides upstream within the ORF
        if i - orf_start >= 20:
            # Extract gRNA sequence (20 nucleotides upstream of PAM)
            grna_start = i - 20
            grna_sequence = dnaseq[grna_start:i]
            
            # Verify the gRNA is entirely within the ORF
            if grna_start >= orf_start:
                forward_grna_pam_pairs.append((i, grna_sequence, pam_sequence))

# Find PAM sequences on the reverse strand (CCN) within the ORF
reverse_grna_pam_pairs = []

# Loop through ORF region looking for CC patterns (reverse strand PAM)
for i in range(orf_start, orf_end - 1):
    if i + 2 > orf_end:
        continue
        
    if dnaseq[i] == 'c' and dnaseq[i+1] == 'c':
        pam_sequence_reverse_strand = dnaseq[i] + dnaseq[i+1] + dnaseq[i+2]
        
        # Check if there are at least 20 nucleotides downstream within the ORF
        if orf_end - (i + 2) >= 20:
            target_start = i + 3
            target_end = target_start + 20
            
            if target_end <= orf_end + 1:
                target_sequence = dnaseq[target_start:target_end]
                grna_sequence = reverse_complement(target_sequence)
                pam_sequence = reverse_complement(pam_sequence_reverse_strand)
                
                reverse_grna_pam_pairs.append((i, grna_sequence, pam_sequence))

# Combine all gRNAs
all_grnas = []
if forward_grna_pam_pairs:
    all_grnas.extend(forward_grna_pam_pairs)
if reverse_grna_pam_pairs:
    all_grnas.extend(reverse_grna_pam_pairs)

# Check GC content for the first gRNA
if all_grnas:
    first_grna_info = all_grnas[0]
    pam_position, grna_sequence, pam_sequence = first_grna_info
    
    # Calculate GC content
    gc_content = calculate_gc_content(grna_sequence)
    
    print(f"First gRNA Sequence Analysis:")
    print(f"gRNA Sequence: {grna_sequence}")
    print(f"PAM Sequence: {pam_sequence}")
    print(f"PAM Position: {pam_position}")
    print(f"GC Content: {gc_content:.2f}%")
    print(f"G Count: {grna_sequence.count('G')}")
    print(f"C Count: {grna_sequence.count('C')}")
    print(f"Total Length: {len(grna_sequence)} bases")
    
    # Check if GC content is optimal
    if 40 <= gc_content <= 60:
        print("✓ GC content is OPTIMAL (40-60%) for CRISPR efficiency")
    elif gc_content < 40:
        print("✗ GC content is TOO LOW (<40%) - may reduce CRISPR efficiency")
    else:
        print("✗ GC content is TOO HIGH (>60%) - may reduce CRISPR efficiency")
        
    # Additional GC content information
    print(f"\nGC Content Guidelines:")
    print("- Optimal range: 40-60%")
    print("- Too low GC (<40%): May reduce gRNA stability and binding affinity")
    print("- Too high GC (>60%): May increase off-target effects")
    
else:
    print("No gRNA sequences found within the ORF.")


First gRNA Sequence Analysis:
gRNA Sequence: tacttaccccatctttcctc
PAM Sequence: ggg
PAM Position: 400
GC Content: 0.00%
G Count: 0
C Count: 0
Total Length: 20 bases
✗ GC content is TOO LOW (<40%) - may reduce CRISPR efficiency

GC Content Guidelines:
- Optimal range: 40-60%
- Too low GC (<40%): May reduce gRNA stability and binding affinity
- Too high GC (>60%): May increase off-target effects



## Stretch goals related to general text processing

1. Make a simple text cipher. For each letter of the alphabet, assign a different letter. You'll need to construct an if, else if, else, statement for this. The final else should be a check to see if there's a character that isn't a letter! If you like you can make a program that can deal with spaces and punctuation too. You will also need to create a new string, which is initiated not to contain any characters.

In [20]:
def explicit_cipher(text):
    """
    Version with explicit if-elif-else for each character type
    """
    result = ""  # Initialize empty string
    
    for char in text:
        # Check each letter individually (this is verbose but follows the requirement)
        if char == 'a': result += 'x'
        elif char == 'b': result += 'y'
        elif char == 'c': result += 'z'
        elif char == 'd': result += 'a'
        elif char == 'e': result += 'b'
        elif char == 'f': result += 'c'
        elif char == 'g': result += 'd'
        elif char == 'h': result += 'e'
        elif char == 'i': result += 'f'
        elif char == 'j': result += 'g'
        elif char == 'k': result += 'h'
        elif char == 'l': result += 'i'
        elif char == 'm': result += 'j'
        elif char == 'n': result += 'k'
        elif char == 'o': result += 'l'
        elif char == 'p': result += 'm'
        elif char == 'q': result += 'n'
        elif char == 'r': result += 'o'
        elif char == 's': result += 'p'
        elif char == 't': result += 'q'
        elif char == 'u': result += 'r'
        elif char == 'v': result += 's'
        elif char == 'w': result += 't'
        elif char == 'x': result += 'u'
        elif char == 'y': result += 'v'
        elif char == 'z': result += 'w'
        
        # Uppercase letters
        elif char == 'A': result += 'X'
        elif char == 'B': result += 'Y'
        elif char == 'C': result += 'Z'
        elif char == 'D': result += 'A'
        elif char == 'E': result += 'B'
        elif char == 'F': result += 'C'
        elif char == 'G': result += 'D'
        elif char == 'H': result += 'E'
        elif char == 'I': result += 'F'
        elif char == 'J': result += 'G'
        elif char == 'K': result += 'H'
        elif char == 'L': result += 'I'
        elif char == 'M': result += 'J'
        elif char == 'N': result += 'K'
        elif char == 'O': result += 'L'
        elif char == 'P': result += 'M'
        elif char == 'Q': result += 'N'
        elif char == 'R': result += 'O'
        elif char == 'S': result += 'P'
        elif char == 'T': result += 'Q'
        elif char == 'U': result += 'R'
        elif char == 'V': result += 'S'
        elif char == 'W': result += 'T'
        elif char == 'X': result += 'U'
        elif char == 'Y': result += 'V'
        elif char == 'Z': result += 'W'
        
        # Final else: handle everything that's not a letter
        else:
            result += char  # Keep spaces, punctuation, numbers, etc.
    
    return result

# Test the explicit version
print("Explicit if-elif-else cipher:")
test_text = "Hello World!"
encrypted_explicit = explicit_cipher(test_text)
print(f"Original: {test_text}")
print(f"Encrypted: {encrypted_explicit}")

Explicit if-elif-else cipher:
Original: Hello World!
Encrypted: Ebiil Tloia!


2. Make a shift (Caesar) cipher. In this cipher, each letter is shifted along the alphabet 5 places, with d going to i, s ging to x, z going to e etc. Note that if you have done some python you will know that you could do this using a dictionary, but try to see if you can do it just using for loops and if statements.

In [22]:
def caesar_cipher_5(text):
    """
    Caesar cipher that shifts each letter 5 places forward in the alphabet.
    Uses only for loops and if statements (no dictionaries).
    """
    result = ""  # Initialize empty string
    
    for char in text:
        # Handle uppercase letters (A-Z)
        if 'A' <= char <= 'Z':
            # Convert to number (A=0, B=1, ..., Z=25)
            char_code = ord(char) - ord('A')
            # Shift by 5 and wrap around using modulo 26
            shifted_code = (char_code + 5) % 26
            # Convert back to character
            shifted_char = chr(shifted_code + ord('A'))
            result += shifted_char
            
        # Handle lowercase letters (a-z)
        elif 'a' <= char <= 'z':
            # Convert to number (a=0, b=1, ..., z=25)
            char_code = ord(char) - ord('a')
            # Shift by 5 and wrap around using modulo 26
            shifted_code = (char_code + 5) % 26
            # Convert back to character
            shifted_char = chr(shifted_code + ord('a'))
            result += shifted_char
            
        # Final else: handle non-letter characters
        else:
            result += char  # Keep spaces, punctuation, numbers, etc.
    
    return result

def caesar_decipher_5(text):
    """
    Decrypt text that was encrypted with the Caesar cipher (shift 5)
    """
    result = ""  # Initialize empty string
    
    for char in text:
        # Handle uppercase letters (A-Z)
        if 'A' <= char <= 'Z':
            # Convert to number (A=0, B=1, ..., Z=25)
            char_code = ord(char) - ord('A')
            # Shift backward by 5 and wrap around using modulo 26
            shifted_code = (char_code - 5) % 26
            # Convert back to character
            shifted_char = chr(shifted_code + ord('A'))
            result += shifted_char
            
        # Handle lowercase letters (a-z)
        elif 'a' <= char <= 'z':
            # Convert to number (a=0, b=1, ..., z=25)
            char_code = ord(char) - ord('a')
            # Shift backward by 5 and wrap around using modulo 26
            shifted_code = (char_code - 5) % 26
            # Convert back to character
            shifted_char = chr(shifted_code + ord('a'))
            result += shifted_char
            
        # Final else: handle non-letter characters
        else:
            result += char  # Keep spaces, punctuation, numbers, etc.
    
    return result

# Test the cipher
if __name__ == "__main__":
    # Test cases to verify the shifting
    test_messages = [
        "Hello World!",
        "Programming is fun!",
        "The quick brown fox jumps over the lazy dog.",
        "abcxyz ABCXYZ",  # Edge cases: a->f, z->e, etc.
        "d goes to i, s goes to x, z goes to e"  # Your specific examples
    ]
    
    for message in test_messages:
        print(f"Original:  {message}")
        encrypted = caesar_cipher_5(message)
        print(f"Encrypted: {encrypted}")
        decrypted = caesar_decipher_5(encrypted)
        print(f"Decrypted: {decrypted}")
        print()
    
    # Verify the specific examples you mentioned
    print("Specific examples verification:")
    examples = {
        'd': 'i',  # d should go to i
        's': 'x',  # s should go to x  
        'z': 'e',  # z should go to e
        'D': 'I',  # uppercase D should go to I
        'S': 'X',  # uppercase S should go to X
        'Z': 'E'   # uppercase Z should go to E
    }
    
    for original, expected in examples.items():
        result = caesar_cipher_5(original)
        print(f"{original} -> {result} (expected: {expected}) {'✓' if result == expected else '✗'}")


Original:  Hello World!
Encrypted: Mjqqt Btwqi!
Decrypted: Hello World!

Original:  Programming is fun!
Encrypted: Uwtlwfrrnsl nx kzs!
Decrypted: Programming is fun!

Original:  The quick brown fox jumps over the lazy dog.
Encrypted: Ymj vznhp gwtbs ktc ozrux tajw ymj qfed itl.
Decrypted: The quick brown fox jumps over the lazy dog.

Original:  abcxyz ABCXYZ
Encrypted: fghcde FGHCDE
Decrypted: abcxyz ABCXYZ

Original:  d goes to i, s goes to x, z goes to e
Encrypted: i ltjx yt n, x ltjx yt c, e ltjx yt j
Decrypted: d goes to i, s goes to x, z goes to e

Specific examples verification:
d -> i (expected: i) ✓
s -> x (expected: x) ✓
z -> e (expected: e) ✓
D -> I (expected: I) ✓
S -> X (expected: X) ✓
Z -> E (expected: E) ✓
